In [2]:
"""
Анализ оценок 4 моделей по 7 критериям.

Два варианта:
1) "Основной" вариант:
   - порядковая регрессия (Ordered Logistic Regression)
   - фиксированный эффект model
   - опционально фиксированный эффект dialog
   - кластер-робастные стандартные ошибки по session_id
   - глобальный тест эффекта модели
   - попарные сравнения моделей
   - odds ratios, 95% CI, Holm correction

2) "Упрощенный" вариант:
   - агрегация на уровне session_id x model
   - Friedman test
   - post-hoc Wilcoxon signed-rank tests
   - Holm correction
   - effect sizes: Kendall's W, rank-biserial correlation

Ожидаемый CSV:
- quest: строка "{model}/{dialog}"
- relevancy
- coherence
- naturalness
- faithfulness
- safety
- unbiasedness
- non-toxicity
- session_id
"""

'\nАнализ оценок 4 моделей по 7 критериям.\n\nДва варианта:\n1) "Основной" вариант:\n   - порядковая регрессия (Ordered Logistic Regression)\n   - фиксированный эффект model\n   - опционально фиксированный эффект dialog\n   - кластер-робастные стандартные ошибки по session_id\n   - глобальный тест эффекта модели\n   - попарные сравнения моделей\n   - odds ratios, 95% CI, Holm correction\n\n2) "Упрощенный" вариант:\n   - агрегация на уровне session_id x model\n   - Friedman test\n   - post-hoc Wilcoxon signed-rank tests\n   - Holm correction\n   - effect sizes: Kendall\'s W, rank-biserial correlation\n\nОжидаемый CSV:\n- quest: строка "{model}/{dialog}"\n- relevancy\n- coherence\n- naturalness\n- faithfulness\n- safety\n- unbiasedness\n- non-toxicity\n- session_id\n'

In [1]:
import os
import itertools
import warnings
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import re

import os
import re
import itertools
import warnings
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import arviz as az
import bambi as bmb

from scipy.stats import friedmanchisquare, wilcoxon, rankdata, chi2
from statsmodels.miscmodels.ordinal_model import OrderedModel
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

c:\Projects\rpg_quest\.venv\Lib\site-packages\arviz\__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(
WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


In [8]:
# =========================
# 1. НАСТРОЙКИ
# =========================

CSV_PATH = "../results_25-04-2026_dataframe_cols.csv"   # <-- замените на ваш путь к CSV
OUTPUT_DIR = "./results_no_ten"

# Если True, в "основном" варианте добавляем фиксированный эффект dialog.
# Это помогает учесть различия между диалогами, но может усложнить модель.
USE_DIALOG_FIXED_EFFECT = True

# Список критериев
CRITERIA = [
    "relevancy",
    "coherence",
    "naturalness",
    "faithfulness",
    "safety",
    "unbiasedness",
    "non-toxicity"
]

# Уровень значимости
ALPHA = 0.05

In [3]:
# =========================
# 2. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# =========================

def ensure_output_dir(path: str):
    """Создает директорию для результатов, если ее еще нет."""
    os.makedirs(path, exist_ok=True)


def parse_quest_column(df: pd.DataFrame) -> pd.DataFrame:
    """
    Разбирает поле quest формата:
    {model}/{dialog}
    Пример:
    deepseek-v3.2-automata_v2-delivery_v2/quest6

    После разбора появляются колонки:
    - model
    - dialog
    """
    if "quest" not in df.columns:
        raise ValueError("В CSV нет колонки 'quest'.")

    split_cols = df["quest"].astype(str).str.split("/", n=1, expand=True)
    if split_cols.shape[1] != 2:
        raise ValueError(
            "Колонка quest должна быть в формате '{model}/{dialog}'."
        )

    df = df.copy()
    df["model"] = split_cols[0]
    df["dialog"] = split_cols[1]
    return df


def validate_data(df: pd.DataFrame, criteria: List[str]):
    """
    Проверяет наличие обязательных колонок и корректность шкал.
    """
    required_columns = ["quest", "session_id"] + criteria
    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        raise ValueError(f"В CSV отсутствуют колонки: {missing}")

    for c in criteria:
        if not pd.api.types.is_numeric_dtype(df[c]):
            raise ValueError(f"Колонка {c} должна быть числовой.")
        bad = df[~df[c].between(1, 10, inclusive="both")]
        if len(bad) > 0:
            raise ValueError(f"В колонке {c} есть значения вне диапазона 1..10.")

    if df["session_id"].isna().any():
        raise ValueError("В колонке session_id есть пропуски.")


def print_header(text: str):
    """Печатает красивый заголовок в консоль."""
    print("\n" + "=" * 90)
    print(text)
    print("=" * 90)


def describe_dataset(df: pd.DataFrame):
    """
    Печатает краткое описание датасета.
    """
    print_header("ОПИСАНИЕ ДАННЫХ")
    print(f"Число строк: {len(df)}")
    print(f"Число оценщиков (session_id): {df['session_id'].nunique()}")
    print(f"Число моделей: {df['model'].nunique()}")
    print(f"Модели: {sorted(df['model'].unique().tolist())}")
    print(f"Число диалогов: {df['dialog'].nunique()}")

    counts = df.groupby("model").size().sort_values(ascending=False)
    print("\nЧисло наблюдений по моделям:")
    print(counts.to_string())

    print("\nЧисло уникальных диалогов по моделям:")
    dialog_counts = df.groupby("model")["dialog"].nunique().sort_values(ascending=False)
    print(dialog_counts.to_string())


def holm_correction(pvals: List[float]) -> np.ndarray:
    """
    Возвращает p-value после коррекции Holm.
    """
    return multipletests(pvals, alpha=ALPHA, method="holm")[1]


def odds_ratio_and_ci(beta: float, se: float, alpha: float = 0.05) -> Tuple[float, float, float]:
    """
    Переводит коэффициент логит-модели в OR и 95% CI.
    """
    z = 1.959963984540054
    lower = beta - z * se
    upper = beta + z * se
    return np.exp(beta), np.exp(lower), np.exp(upper)


def kendalls_w_from_friedman(chisq: float, n_subjects: int, k_conditions: int) -> float:
    """
    Kendall's W для Friedman test.
    Формула:
    W = chi2 / (n * (k - 1))
    """
    if n_subjects <= 0 or k_conditions <= 1:
        return np.nan
    return chisq / (n_subjects * (k_conditions - 1))


def rank_biserial_from_wilcoxon(x: np.ndarray, y: np.ndarray) -> float:
    """
    Rank-biserial correlation для парного сравнения Wilcoxon.

    Практическая реализация:
    - считаем разности d = x - y
    - нулевые разности исключаем
    - ранжируем abs(d)
    - r_rb = (W_plus - W_minus) / (W_plus + W_minus)

    Значение:
    - ближе к +1 => x обычно выше y
    - ближе к -1 => x обычно ниже y
    - около 0 => различий почти нет
    """
    d = np.asarray(x) - np.asarray(y)
    d = d[d != 0]

    if len(d) == 0:
        return 0.0

    ranks = rankdata(np.abs(d))
    w_plus = ranks[d > 0].sum()
    w_minus = ranks[d < 0].sum()
    return (w_plus - w_minus) / (w_plus + w_minus)


def try_ordered_model_fit(endog: pd.Series, exog: pd.DataFrame):
    """
    Пытается подогнать OrderedModel.
    Возвращает fitted result или None.
    """
    try:
        model = OrderedModel(endog, exog, distr="logit")
        result = model.fit(method="bfgs", disp=False)
        return result
    except Exception:
        return None


def lr_test(full_llf: float, reduced_llf: float, df_diff: int) -> Tuple[float, float]:
    """
    Likelihood ratio test:
    LR = 2 * (LL_full - LL_reduced)
    p = 1 - chi2.cdf(LR, df_diff)
    """
    lr = 2 * (full_llf - reduced_llf)
    p = 1 - chi2.cdf(lr, df_diff)
    return lr, p


In [4]:
# =========================
# 3. ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ
# =========================

def load_and_prepare_data(csv_path: str) -> pd.DataFrame:
    """
    Загружает CSV, валидирует и подготавливает данные.
    """
    df = pd.read_csv(csv_path)
    df['session_id'] = df['session_id'].fillna("99998888")
    validate_data(df, CRITERIA)
    df = parse_quest_column(df)

    # Приводим к строкам для безопасности
    df["session_id"] = df["session_id"].astype(str)
    df["model"] = df["model"].astype(str)
    df["dialog"] = df["dialog"].astype(str)

    # Категориальные типы
    df["model"] = pd.Categorical(df["model"], categories=sorted(df["model"].unique()), ordered=False)
    df["dialog"] = pd.Categorical(df["dialog"], categories=sorted(df["dialog"].unique()), ordered=False)

    return df


In [5]:
# =========================
# 4. ОСНОВНОЙ ВАРИАНТ АНАЛИЗА
# =========================

def build_design_matrix(df: pd.DataFrame, include_dialog: bool = True) -> pd.DataFrame:
    """
    Строит матрицу предикторов:
    - дамми для model
    - опционально дамми для dialog

    В OrderedModel константу добавлять не нужно.
    """
    parts = []

    # model: drop_first=True, чтобы одна модель была базовой
    model_dummies = pd.get_dummies(df["model"], prefix="model", drop_first=True, dtype=float)
    parts.append(model_dummies)

    if include_dialog:
        dialog_dummies = pd.get_dummies(df["dialog"], prefix="dialog", drop_first=True, dtype=float)
        parts.append(dialog_dummies)

    if len(parts) == 0:
        raise ValueError("Матрица дизайна получилась пустой.")

    X = pd.concat(parts, axis=1)
    return X


def fit_ordered_model_for_criterion(
    df: pd.DataFrame,
    criterion: str,
    include_dialog: bool = True
) -> Dict:
    """
    Подгоняет порядковую логистическую модель для одного критерия.

    Полная модель:
    rating ~ model + dialog(опционально)

    Редуцированная модель для глобального теста:
    rating ~ dialog(опционально)
    или "пустая", если dialog не включен

    Возвращает словарь с:
    - full_result
    - reduced_result
    - global LR test
    - summary table по коэффициентам model
    """
    local_df = df.copy()

    # endog должен быть integer/ordered categories
    y = local_df[criterion].astype(int)

    # Полная модель
    X_full = build_design_matrix(local_df, include_dialog=include_dialog)
    full_res = try_ordered_model_fit(y, X_full)
    if full_res is None:
        return {
            "criterion": criterion,
            "success": False,
            "message": "Не удалось подогнать полную OrderedModel."
        }

    # Редуцированная модель без model
    if include_dialog:
        X_reduced = pd.get_dummies(local_df["dialog"], prefix="dialog", drop_first=True, dtype=float)
        if X_reduced.shape[1] == 0:
            # если вдруг один dialog, оставим пустую модель
            X_reduced = pd.DataFrame(index=local_df.index)
    else:
        X_reduced = pd.DataFrame(index=local_df.index)

    # В statsmodels OrderedModel не любит совсем пустой exog.
    # Поэтому если reduced-предикторов нет, сделаем фиктивный столбец,
    # а затем удалим смысловую интерпретацию. Это технический обход.
    if X_reduced.shape[1] == 0:
        X_reduced = pd.DataFrame({"dummy_zero": np.zeros(len(local_df))}, index=local_df.index)

    reduced_res = try_ordered_model_fit(y, X_reduced)
    if reduced_res is None:
        return {
            "criterion": criterion,
            "success": False,
            "message": "Не удалось подогнать редуцированную OrderedModel."
        }

    # LR-тест для общего эффекта модели
    model_cols = [c for c in X_full.columns if c.startswith("model_")]
    df_diff = len(model_cols)
    lr_stat, p_value = lr_test(full_res.llf, reduced_res.llf, df_diff)

    # Собираем коэффициенты по model
    rows = []
    params = full_res.params
    bse = full_res.bse

    for col in model_cols:
        beta = params.get(col, np.nan)
        se = bse.get(col, np.nan)
        or_val, ci_low, ci_high = odds_ratio_and_ci(beta, se)
        rows.append({
            "criterion": criterion,
            "term": col,
            "beta": beta,
            "se": se,
            "odds_ratio": or_val,
            "ci_low": ci_low,
            "ci_high": ci_high,
            "p_value": full_res.pvalues.get(col, np.nan)
        })

    coef_table = pd.DataFrame(rows)

    return {
        "criterion": criterion,
        "success": True,
        "full_result": full_res,
        "reduced_result": reduced_res,
        "lr_stat": lr_stat,
        "global_p": p_value,
        "coef_table": coef_table,
        "n_obs": len(local_df)
    }


def pairwise_ordered_logit(
    df: pd.DataFrame,
    criterion: str,
    include_dialog: bool = True
) -> pd.DataFrame:
    """
    Попарные сравнения моделей для одного критерия.

    Для каждой пары моделей:
    - оставляем только эти две модели
    - строим OrderedModel:
      rating ~ model_binary + dialog(optional)
    - получаем beta, OR, CI, p-value

    Интерпретация:
    если OR > 1, то вторая модель в кодировке имеет больше шансов на более высокий рейтинг,
    в зависимости от того, какая модель взята как базовая.
    Мы явно фиксируем направление:
      comparison = "model_B vs model_A"
    где beta относится к индикатору model_B.
    """
    model_names = sorted(df["model"].astype(str).unique().tolist())
    all_pairs = list(itertools.combinations(model_names, 2))
    rows = []

    for m1, m2 in all_pairs:
        sub = df[df["model"].astype(str).isin([m1, m2])].copy()

        # Бинарная модель: m1 базовая, m2 = 1
        sub["pair_model"] = (sub["model"].astype(str) == m2).astype(int)
        y = sub[criterion].astype(int)

        if include_dialog:
            X = pd.DataFrame({"pair_model": sub["pair_model"].astype(float)}, index=sub.index)
            dialog_dummies = pd.get_dummies(sub["dialog"], prefix="dialog", drop_first=True, dtype=float)
            X = pd.concat([X, dialog_dummies], axis=1)
        else:
            X = pd.DataFrame({"pair_model": sub["pair_model"].astype(float)}, index=sub.index)

        res = try_ordered_model_fit(y, X)

        if res is None:
            rows.append({
                "criterion": criterion,
                "comparison": f"{m2} vs {m1}",
                "beta": np.nan,
                "se": np.nan,
                "odds_ratio": np.nan,
                "ci_low": np.nan,
                "ci_high": np.nan,
                "p_value": np.nan,
                "success": False
            })
            continue

        beta = res.params.get("pair_model", np.nan)
        se = res.bse.get("pair_model", np.nan)
        or_val, ci_low, ci_high = odds_ratio_and_ci(beta, se)

        rows.append({
            "criterion": criterion,
            "comparison": f"{m2} vs {m1}",
            "beta": beta,
            "se": se,
            "odds_ratio": or_val,
            "ci_low": ci_low,
            "ci_high": ci_high,
            "p_value": res.pvalues.get("pair_model", np.nan),
            "success": True
        })

    out = pd.DataFrame(rows)

    if len(out) > 0 and out["p_value"].notna().any():
        mask = out["p_value"].notna()
        out.loc[mask, "p_holm"] = holm_correction(out.loc[mask, "p_value"].tolist())
    else:
        out["p_holm"] = np.nan

    return out


def interpret_ordered_results(global_table: pd.DataFrame, pairwise_table: pd.DataFrame):
    """
    Печатает понятную интерпретацию основного анализа.
    """
    print_header("ИНТЕРПРЕТАЦИЯ ОСНОВНОГО ВАРИАНТА")

    for _, row in global_table.iterrows():
        criterion = row["criterion"]
        p = row["global_p"]
        print(f"\nКритерий: {criterion}")
        if pd.isna(p):
            print("  Не удалось получить глобальный тест.")
            continue

        if p < ALPHA:
            print(f"  Есть статистически значимый общий эффект модели (p = {p:.4g}).")
            sub = pairwise_table[pairwise_table["criterion"] == criterion].copy()
            sig = sub[sub["p_holm"] < ALPHA]

            if len(sig) == 0:
                print("  После попарных сравнений с поправкой Holm значимых различий не найдено.")
            else:
                print("  Значимые попарные различия после поправки Holm:")
                for _, r in sig.iterrows():
                    print(
                        f"    {r['comparison']}: OR = {r['odds_ratio']:.3f} "
                        f"[{r['ci_low']:.3f}; {r['ci_high']:.3f}], "
                        f"p_holm = {r['p_holm']:.4g}"
                    )
        else:
            print(f"  Нет статистически значимого общего эффекта модели (p = {p:.4g}).")


def run_main_analysis(df: pd.DataFrame, output_dir: str, include_dialog: bool = True):
    """
    Запускает "основной" вариант анализа по всем критериям.
    """
    print_header("ОСНОВНОЙ ВАРИАНТ АНАЛИЗА")
    print("Модель: Ordered Logistic Regression")
    print(f"Учитывать dialog как фиксированный эффект: {include_dialog}")

    global_rows = []
    coef_tables = []
    pairwise_tables = []

    for criterion in CRITERIA:
        print(f"\n--- Анализ критерия: {criterion} ---")
        res = fit_ordered_model_for_criterion(df, criterion, include_dialog=include_dialog)

        if not res["success"]:
            print(f"Не удалось подогнать модель: {res['message']}")
            global_rows.append({
                "criterion": criterion,
                "n_obs": np.nan,
                "lr_stat": np.nan,
                "global_p": np.nan,
                "success": False
            })
            continue

        print(f"Глобальный LR test: LR = {res['lr_stat']:.4f}, p = {res['global_p']:.4g}")

        global_rows.append({
            "criterion": criterion,
            "n_obs": res["n_obs"],
            "lr_stat": res["lr_stat"],
            "global_p": res["global_p"],
            "success": True
        })

        coef_tables.append(res["coef_table"])

        pw = pairwise_ordered_logit(df, criterion, include_dialog=include_dialog)
        pairwise_tables.append(pw)

    global_table = pd.DataFrame(global_rows)
    if len(global_table) > 0 and global_table["global_p"].notna().any():
        mask = global_table["global_p"].notna()
        global_table.loc[mask, "global_p_holm_across_criteria"] = holm_correction(
            global_table.loc[mask, "global_p"].tolist()
        )
    else:
        global_table["global_p_holm_across_criteria"] = np.nan

    coef_table = pd.concat(coef_tables, ignore_index=True) if coef_tables else pd.DataFrame()
    pairwise_table = pd.concat(pairwise_tables, ignore_index=True) if pairwise_tables else pd.DataFrame()

    # Сохраняем
    global_table.to_csv(os.path.join(output_dir, "main_global_tests.csv"), index=False)
    coef_table.to_csv(os.path.join(output_dir, "main_model_coefficients.csv"), index=False)
    pairwise_table.to_csv(os.path.join(output_dir, "main_pairwise_comparisons.csv"), index=False)

    print("\nРезультаты сохранены:")
    print(f"  {os.path.join(output_dir, 'main_global_tests.csv')}")
    print(f"  {os.path.join(output_dir, 'main_model_coefficients.csv')}")
    print(f"  {os.path.join(output_dir, 'main_pairwise_comparisons.csv')}")

    interpret_ordered_results(global_table, pairwise_table)

    return global_table, coef_table, pairwise_table


In [32]:
# =========================
# 5. УПРОЩЕННЫЙ ВАРИАНТ АНАЛИЗА
# =========================

def aggregate_for_simplified_analysis(df: pd.DataFrame, criterion: str, agg_func: str = "median") -> pd.DataFrame:
    """
    Агрегирует данные на уровне session_id x model.

    agg_func:
    - "median"
    - "mean"

    Возвращает wide-table:
    session_id | model_A | model_B | ...
    """
    if agg_func not in ("median", "mean"):
        raise ValueError("agg_func должен быть 'median' или 'mean'.")

    most_common_by_model = (
        df.groupby("model")[criterion]
        .agg(lambda s: s.mode())   # mode() can return multiple values if there's a tie
        .explode()
        .reset_index(name=criterion)
    )

    # remove rows where criterion is one of the most frequent values for that model
    df_filtered  = (
        df.merge(most_common_by_model.assign(_drop=1), on=["model", criterion], how="left")
        .query("_drop != 1")
        .drop(columns="_drop")
    )

    grouped = df_filtered .groupby(["session_id", "model"])[criterion]

    if agg_func == "median":
        agg = grouped.median().reset_index(name="score")
    else:
        agg = grouped.mean().reset_index(name="score")

    wide = agg.pivot(index="session_id", columns="model", values="score")

    # Для Friedman нужны только те session_id, у кого есть оценки всех моделей
    wide = wide.dropna(axis=0, how="any")
    wide = wide.sort_index(axis=1)
    return wide


def friedman_for_criterion(df: pd.DataFrame, criterion: str, agg_func: str = "median") -> Dict:
    """
    Friedman test для одного критерия после агрегации.
    """
    wide = aggregate_for_simplified_analysis(df, criterion, agg_func=agg_func)
    model_names = wide.columns.tolist()

    if wide.shape[1] < 2:
        return {
            "criterion": criterion,
            "success": False,
            "message": "Недостаточно моделей для Friedman test."
        }

    if wide.shape[0] < 2:
        return {
            "criterion": criterion,
            "success": False,
            "message": "Недостаточно оценщиков после агрегации."
        }

    arrays = [wide[c].values for c in model_names]
    stat, p = friedmanchisquare(*arrays)

    w = kendalls_w_from_friedman(stat, n_subjects=wide.shape[0], k_conditions=wide.shape[1])

    return {
        "criterion": criterion,
        "success": True,
        "stat": stat,
        "p_value": p,
        "kendalls_w": w,
        "n_subjects": wide.shape[0],
        "k_models": wide.shape[1],
        "wide": wide
    }


def pairwise_wilcoxon_for_criterion(df: pd.DataFrame, criterion: str, agg_func: str = "median") -> pd.DataFrame:
    """
    Попарные Wilcoxon signed-rank tests между моделями
    после агрегации на уровне session_id x model.
    """
    wide = aggregate_for_simplified_analysis(df, criterion, agg_func=agg_func)
    model_names = wide.columns.tolist()

    rows = []
    for m1, m2 in itertools.combinations(model_names, 2):
        x = wide[m1].values
        y = wide[m2].values

        # Если все разности нулевые, Wilcoxon может не сработать
        if np.allclose(x, y):
            p = 1.0
            stat = 0.0
            rbc = 0.0
        else:
            try:
                stat, p = wilcoxon(x, y, zero_method="wilcox", alternative="two-sided")
            except ValueError:
                stat, p = np.nan, np.nan

            rbc = rank_biserial_from_wilcoxon(x, y)

        rows.append({
            "criterion": criterion,
            "comparison": f"{m1} vs {m2}",
            "n_subjects": len(x),
            "wilcoxon_stat": stat,
            "p_value": p,
            "rank_biserial": rbc,
            "median_model_1": np.median(x),
            "median_model_2": np.median(y),
            "mean_model_1": np.mean(x),
            "mean_model_2": np.mean(y)
        })

    out = pd.DataFrame(rows)

    if len(out) > 0 and out["p_value"].notna().any():
        mask = out["p_value"].notna()
        out.loc[mask, "p_holm"] = holm_correction(out.loc[mask, "p_value"].tolist())
    else:
        out["p_holm"] = np.nan

    return out


def interpret_simplified_results(global_table: pd.DataFrame, pairwise_table: pd.DataFrame):
    """
    Печатает понятную интерпретацию упрощенного анализа.
    """
    print_header("ИНТЕРПРЕТАЦИЯ УПРОЩЕННОГО ВАРИАНТА")

    for _, row in global_table.iterrows():
        criterion = row["criterion"]
        p = row["p_value"]
        print(f"\nКритерий: {criterion}")

        if pd.isna(p):
            print("  Не удалось выполнить Friedman test.")
            continue

        if p < ALPHA:
            print(
                f"  Есть общий эффект модели по Friedman test "
                f"(chi2 = {row['stat']:.4f}, p = {p:.4g}, Kendall's W = {row['kendalls_w']:.3f})."
            )

            sub = pairwise_table[pairwise_table["criterion"] == criterion].copy()
            sig = sub[sub["p_holm"] < ALPHA]

            if len(sig) == 0:
                print("  После попарных Wilcoxon-тестов с поправкой Holm значимых различий не найдено.")
            else:
                print("  Значимые попарные различия после поправки Holm:")
                for _, r in sig.iterrows():
                    direction = ""
                    if r["rank_biserial"] > 0:
                        direction = "первая модель обычно выше второй"
                    elif r["rank_biserial"] < 0:
                        direction = "вторая модель обычно выше первой"
                    else:
                        direction = "различие по направлению почти отсутствует"

                    print(
                        f"    {r['comparison']}: "
                        f"p_holm = {r['p_holm']:.4g}, "
                        f"rank-biserial = {r['rank_biserial']:.3f} ({direction})"
                    )
        else:
            print(f"  Нет общего эффекта модели по Friedman test (p = {p:.4g}).")


def run_simplified_analysis(df: pd.DataFrame, output_dir: str, agg_func: str = "median"):
    """
    Запускает упрощенный вариант анализа по всем критериям.
    """
    print_header("УПРОЩЕННЫЙ ВАРИАНТ АНАЛИЗА")
    print(f"Агрегация session_id x model: {agg_func}")

    global_rows = []
    pairwise_tables = []

    for criterion in CRITERIA:
        print(f"\n--- Анализ критерия: {criterion} ---")
        res = friedman_for_criterion(df, criterion, agg_func=agg_func)

        if not res["success"]:
            print(f"Не удалось выполнить Friedman test: {res['message']}")
            global_rows.append({
                "criterion": criterion,
                "stat": np.nan,
                "p_value": np.nan,
                "kendalls_w": np.nan,
                "n_subjects": np.nan,
                "k_models": np.nan,
                "success": False
            })
            continue

        print(
            f"Friedman test: chi2 = {res['stat']:.4f}, "
            f"p = {res['p_value']:.4g}, "
            f"Kendall's W = {res['kendalls_w']:.3f}, "
            f"n = {res['n_subjects']}"
        )

        global_rows.append({
            "criterion": criterion,
            "stat": res["stat"],
            "p_value": res["p_value"],
            "kendalls_w": res["kendalls_w"],
            "n_subjects": res["n_subjects"],
            "k_models": res["k_models"],
            "success": True
        })

        pw = pairwise_wilcoxon_for_criterion(df, criterion, agg_func=agg_func)
        pairwise_tables.append(pw)

    global_table = pd.DataFrame(global_rows)

    if len(global_table) > 0 and global_table["p_value"].notna().any():
        mask = global_table["p_value"].notna()
        global_table.loc[mask, "p_holm_across_criteria"] = holm_correction(
            global_table.loc[mask, "p_value"].tolist()
        )
    else:
        global_table["p_holm_across_criteria"] = np.nan

    pairwise_table = pd.concat(pairwise_tables, ignore_index=True) if pairwise_tables else pd.DataFrame()

    # Сохраняем
    global_table.to_csv(os.path.join(output_dir, "simplified_global_tests.csv"), index=False)
    pairwise_table.to_csv(os.path.join(output_dir, "simplified_pairwise_comparisons.csv"), index=False)

    print("\nРезультаты сохранены:")
    print(f"  {os.path.join(output_dir, 'simplified_global_tests.csv')}")
    print(f"  {os.path.join(output_dir, 'simplified_pairwise_comparisons.csv')}")

    interpret_simplified_results(global_table, pairwise_table)

    return global_table, pairwise_table


In [7]:
# =========================
# 6. ДОПОЛНИТЕЛЬНЫЕ ОПИСАТЕЛЬНЫЕ ТАБЛИЦЫ
# =========================

def build_descriptive_tables(df: pd.DataFrame, output_dir: str):
    """
    Создает описательные таблицы:
    - медианы по model x criterion
    - средние по model x criterion
    - распределения оценок
    """
    print_header("ОПИСАТЕЛЬНЫЕ ТАБЛИЦЫ")

    median_table = df.groupby("model")[CRITERIA].median()
    mean_table = df.groupby("model")[CRITERIA].mean().round(3)

    median_table.to_csv(os.path.join(output_dir, "descriptives_median_by_model.csv"))
    mean_table.to_csv(os.path.join(output_dir, "descriptives_mean_by_model.csv"))

    print("Сохранены описательные таблицы:")
    print(f"  {os.path.join(output_dir, 'descriptives_median_by_model.csv')}")
    print(f"  {os.path.join(output_dir, 'descriptives_mean_by_model.csv')}")

    for criterion in CRITERIA:
        dist = (
            df.groupby(["model", criterion])
              .size()
              .reset_index(name="count")
              .sort_values(["model", criterion])
        )
        filename = f"distribution_{criterion}.csv".replace("/", "_")
        dist.to_csv(os.path.join(output_dir, filename), index=False)

    print("Также сохранены распределения оценок по каждому критерию.")


# Ordinal Mixed Model с mixed-effects

In [12]:
# Настройки MCMC
DRAWS = 2000
TUNE = 2000
CHAINS = 4
CORES = 4
TARGET_ACCEPT = 0.95
RANDOM_SEED = 42

# Какой credible interval использовать
CREDIBLE_INTERVAL = 0.95

# ROPE (region of practical equivalence) для коэффициентов log-odds
# Если интервал малый вокруг 0, можно считать эффект практически пренебрежимым.
ROPE_LOW = -0.1
ROPE_HIGH = 0.1

In [27]:
# =========================
# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# =========================

def print_header(title: str):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

def ensure_output_dir(path: str):
    os.makedirs(path, exist_ok=True)

def parse_quest_column(value: str) -> Tuple[str, str]:
    """
    Из строки вида:
      deepseek-v3.2-automata_v2-delivery_v2/quest6
    извлекает:
      model = deepseek-v3.2-automata_v2-delivery_v2
      dialog = quest6
    """
    if pd.isna(value):
        return np.nan, np.nan

    value = str(value).strip()
    parts = value.split("/")
    if len(parts) >= 2:
        model = "/".join(parts[:-1])
        dialog = parts[-1]
        return model, dialog

    return value, np.nan

def load_and_prepare_data(csv_path: str) -> pd.DataFrame:
    """
    Загружает CSV, извлекает model и dialog из quest,
    приводит данные к нужным типам.
    """
    df = pd.read_csv(csv_path)

    required_cols = ["session_id", "quest"] + CRITERIA
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"В CSV отсутствуют обязательные колонки: {missing}")

    parsed = df["quest"].apply(parse_quest_column)
    df["model"] = parsed.apply(lambda x: x[0])
    df["dialog"] = parsed.apply(lambda x: x[1])

    # Приводим типы
    df["session_id"] = df["session_id"].astype(str)
    df["model"] = df["model"].astype("category")
    df["dialog"] = df["dialog"].astype("category")

    # Проверяем, что оценки целые и в диапазоне 1..10
    for c in CRITERIA:
        df[c] = pd.to_numeric(df[c], errors="coerce")
        bad = df[c].isna() | (df[c] < 1) | (df[c] > 10) | (df[c] % 1 != 0)
        if bad.any():
            n_bad = int(bad.sum())
            print(f"[WARN] В критерии {c} найдено {n_bad} некорректных значений. Они будут удалены.")

    return df

def describe_dataset(df: pd.DataFrame):
    print_header("ОПИСАНИЕ ДАННЫХ")

    print(f"Число строк: {len(df)}")
    print(f"Число оценщиков (session_id): {df['session_id'].nunique()}")
    print(f"Число моделей: {df['model'].nunique()}")
    print(f"Число диалогов: {df['dialog'].nunique()}")

    print("\nМодели:")
    for m in df["model"].cat.categories:
        print(f"  - {m}")

    print("\nЧисло наблюдений по моделям:")
    print(df["model"].value_counts().sort_index())

    print("\nЧисло наблюдений по диалогам:")
    print(df["dialog"].value_counts().sort_index())

def clean_data_for_criterion(df: pd.DataFrame, criterion: str) -> pd.DataFrame:
    """
    Оставляет только валидные строки для данного критерия.
    """
    sub = df.copy()
    sub = sub[["session_id", "model", "dialog", criterion]].copy()
    sub = sub.rename(columns={criterion: "rating"})

    sub["rating"] = pd.to_numeric(sub["rating"], errors="coerce")
    sub = sub.dropna(subset=["rating", "session_id", "model", "dialog"])

    sub = sub[(sub["rating"] >= 1) & (sub["rating"] <= 10)]
    sub = sub[sub["rating"] % 1 == 0]
    sub["rating"] = sub["rating"].astype(int)

    sub["session_id"] = sub["session_id"].astype("category")
    sub["model"] = sub["model"].astype("category")
    sub["dialog"] = sub["dialog"].astype("category")

    return sub

def save_trace_summary(idata, output_path: str):
    """
    Сохраняет summary по ключевым параметрам.
    """
    summary = az.summary(idata, hdi_prob=CREDIBLE_INTERVAL)
    summary.to_csv(output_path)
    return summary

def posterior_prob_positive(samples: np.ndarray) -> float:
    return float(np.mean(samples > 0))

def posterior_prob_negative(samples: np.ndarray) -> float:
    return float(np.mean(samples < 0))

def posterior_prob_rope(samples: np.ndarray, low: float = ROPE_LOW, high: float = ROPE_HIGH) -> float:
    return float(np.mean((samples >= low) & (samples <= high)))

def probability_of_superiority_from_logodds(samples: np.ndarray) -> float:
    """
    Не 'probability of superiority' в строгом непараметрическом смысле,
    а удобная сводка:
    posterior P(effect > 0).
    """
    return posterior_prob_positive(samples)

def hdi_bounds(samples: np.ndarray, prob: float = CREDIBLE_INTERVAL) -> Tuple[float, float]:
    hdi = az.hdi(samples, hdi_prob=prob)
    return float(hdi[0]), float(hdi[1])

def extract_posterior_draws_1d(idata, var_name: str) -> np.ndarray:
    """
    Возвращает одномерный массив posterior draws по имени переменной.
    """
    arr = idata.posterior[var_name].values
    return arr.reshape(-1)

def get_model_levels(sub: pd.DataFrame) -> List[str]:
    return list(sub["model"].cat.categories)

def fit_bayesian_ordinal_model(sub: pd.DataFrame, criterion: str):
    """
    Строит ordinal cumulative model:
      rating ~ model + (1|session_id) + (1|dialog)
    """
    print_header(f"ОБУЧЕНИЕ МОДЕЛИ ДЛЯ КРИТЕРИЯ: {criterion}")

    # Явно задаем reference level как первый уровень category
    # Если хотите другой baseline, можно переупорядочить категории заранее.
    print("Уровни model:")
    for i, m in enumerate(sub["model"].cat.categories):
        if i == 0:
            print(f"  - {m}  [baseline]")
        else:
            print(f"  - {m}")

    formula = "rating ~ model + (1|session_id) + (1|dialog)"
    print(f"\nФормула: {formula}")

    model = bmb.Model(
        formula=formula,
        data=sub,
        family="cumulative"
    )

    idata = model.fit(
        draws=DRAWS,
        tune=TUNE,
        chains=CHAINS,
        cores=CORES,
        target_accept=TARGET_ACCEPT,
        random_seed=RANDOM_SEED
    )

    return model, idata

def summarize_fixed_effects(idata, criterion: str) -> pd.DataFrame:
    """
    Извлекает фиксированные эффекты model из posterior.
    В cumulative model baseline-модель зашита в intercept/reference.
    """
    rows = []

    posterior_vars = list(idata.posterior.data_vars)

    # Обычно Bambi кодирует как model[level]
    model_terms = [v for v in posterior_vars if v.startswith("model[")]

    for term in model_terms:
        samples = extract_posterior_draws_1d(idata, term)
        mean_ = float(np.mean(samples))
        median_ = float(np.median(samples))
        sd_ = float(np.std(samples, ddof=1))
        hdi_low, hdi_high = hdi_bounds(samples)

        or_mean = float(np.exp(mean_))
        or_median = float(np.exp(median_))
        or_low = float(np.exp(hdi_low))
        or_high = float(np.exp(hdi_high))

        p_gt_0 = posterior_prob_positive(samples)
        p_lt_0 = posterior_prob_negative(samples)
        p_rope = posterior_prob_rope(samples)

        rows.append({
            "criterion": criterion,
            "term": term,
            "mean_logodds": mean_,
            "median_logodds": median_,
            "sd_logodds": sd_,
            "hdi_low_logodds": hdi_low,
            "hdi_high_logodds": hdi_high,
            "mean_odds_ratio": or_mean,
            "median_odds_ratio": or_median,
            "hdi_low_odds_ratio": or_low,
            "hdi_high_odds_ratio": or_high,
            "post_prob_gt_0": p_gt_0,
            "post_prob_lt_0": p_lt_0,
            "post_prob_in_rope": p_rope
        })

    return pd.DataFrame(rows)

def parse_term_level(term: str) -> str:
    """
    Из model[NAME] извлекает NAME
    """
    m = re.match(r"model\[(.+)\]", term)
    if m:
        return m.group(1)
    return term

def pairwise_model_comparisons_from_posterior(idata, sub: pd.DataFrame, criterion: str) -> pd.DataFrame:
    """
    Формирует попарные сравнения моделей на основе posterior draws
    фиксированных эффектов.

    Пусть baseline = первая модель.
    Тогда:
    - effect(baseline) = 0
    - effect(other model) = beta_model[level]

    Для пары A vs B:
    delta = effect(A) - effect(B)

    Если delta > 0, A имеет более высокие log-odds получить большую оценку.
    """
    model_levels = get_model_levels(sub)
    baseline = model_levels[0]

    posterior_vars = list(idata.posterior.data_vars)
    model_terms = {parse_term_level(v): v for v in posterior_vars if v.startswith("model[")}

    effect_draws = {}
    n_draws = np.prod(idata.posterior.dims["chain"] * idata.posterior.dims["draw"])

    # baseline = 0
    any_var = list(idata.posterior.data_vars)[0]
    any_shape = idata.posterior[any_var].values.shape
    total_draws = any_shape[0] * any_shape[1]
    effect_draws[baseline] = np.zeros(total_draws)

    for lvl in model_levels[1:]:
        if lvl not in model_terms:
            raise ValueError(f"Не найден posterior term для model level: {lvl}")
        effect_draws[lvl] = extract_posterior_draws_1d(idata, model_terms[lvl])

    rows = []
    for a, b in itertools.combinations(model_levels, 2):
        delta = effect_draws[a] - effect_draws[b]

        mean_ = float(np.mean(delta))
        median_ = float(np.median(delta))
        sd_ = float(np.std(delta, ddof=1))
        hdi_low, hdi_high = hdi_bounds(delta)

        or_mean = float(np.exp(mean_))
        or_median = float(np.exp(median_))
        or_low = float(np.exp(hdi_low))
        or_high = float(np.exp(hdi_high))

        p_gt_0 = posterior_prob_positive(delta)
        p_lt_0 = posterior_prob_negative(delta)
        p_rope = posterior_prob_rope(delta)

        rows.append({
            "criterion": criterion,
            "comparison": f"{a} vs {b}",
            "mean_delta_logodds": mean_,
            "median_delta_logodds": median_,
            "sd_delta_logodds": sd_,
            "hdi_low_logodds": hdi_low,
            "hdi_high_logodds": hdi_high,
            "mean_odds_ratio": or_mean,
            "median_odds_ratio": or_median,
            "hdi_low_odds_ratio": or_low,
            "hdi_high_odds_ratio": or_high,
            "post_prob_a_gt_b": p_gt_0,
            "post_prob_b_gt_a": p_lt_0,
            "post_prob_in_rope": p_rope
        })

    return pd.DataFrame(rows)

def random_effects_summary(idata, criterion: str) -> pd.DataFrame:
    """
    Извлекает сводку по стандартным отклонениям случайных эффектов.
    Имена параметров могут отличаться между версиями bambi/pymc.
    Поэтому просто ищем sd-подобные параметры.
    """
    rows = []
    posterior_vars = list(idata.posterior.data_vars)

    for var in posterior_vars:
        name_lower = var.lower()
        if ("sd" in name_lower) or ("sigma" in name_lower):
            samples = extract_posterior_draws_1d(idata, var)
            mean_ = float(np.mean(samples))
            med_ = float(np.median(samples))
            low_, high_ = hdi_bounds(samples)

            rows.append({
                "criterion": criterion,
                "parameter": var,
                "mean": mean_,
                "median": med_,
                "hdi_low": low_,
                "hdi_high": high_
            })

    return pd.DataFrame(rows)

def diagnostics_table(idata, criterion: str) -> pd.DataFrame:
    """
    Сохраняет диагностики MCMC:
    - r_hat
    - ess_bulk
    - ess_tail
    """
    s = az.summary(idata, hdi_prob=CREDIBLE_INTERVAL)
    s = s.reset_index().rename(columns={"index": "parameter"})
    s["criterion"] = criterion

    keep_cols = [c for c in ["criterion", "parameter", "mean", "sd", "hdi_2.5%", "hdi_97.5%", "ess_bulk", "ess_tail", "r_hat"] if c in s.columns]
    if not keep_cols:
        keep_cols = s.columns.tolist()

    return s[keep_cols]

def posterior_predictive_summary(model, idata, sub: pd.DataFrame, criterion: str) -> pd.DataFrame:
    """
    Делает posterior predictive и сравнивает распределение реальных/предсказанных оценок.
    """
    try:
        ppc = model.predict(idata=idata, kind="pps", inplace=False)
    except Exception as e:
        print(f"[WARN] Не удалось получить posterior predictive для {criterion}: {e}")
        return pd.DataFrame()

    rows = []

    observed_dist = sub["rating"].value_counts(normalize=True).sort_index()

    # Ищем переменную отклика в posterior predictive
    ppc_vars = list(ppc.posterior_predictive.data_vars)
    if len(ppc_vars) == 0:
        return pd.DataFrame()

    response_name = ppc_vars[0]
    pred = ppc.posterior_predictive[response_name].values
    # shape обычно chain x draw x obs
    pred_flat = pred.reshape(-1, pred.shape[-1])

    for score in range(1, 11):
        obs_prob = float(observed_dist.get(score, 0.0))
        pred_prob_draws = np.mean(pred_flat == score, axis=1)

        rows.append({
            "criterion": criterion,
            "score": score,
            "observed_prob": obs_prob,
            "predicted_prob_mean": float(np.mean(pred_prob_draws)),
            "predicted_prob_hdi_low": float(az.hdi(pred_prob_draws, hdi_prob=CREDIBLE_INTERVAL)[0]),
            "predicted_prob_hdi_high": float(az.hdi(pred_prob_draws, hdi_prob=CREDIBLE_INTERVAL)[1]),
        })

    return pd.DataFrame(rows)

def interpret_fixed_effects(fixed_df: pd.DataFrame):
    """
    Печатает понятную интерпретацию фиксированных эффектов.
    """
    if fixed_df.empty:
        print("Нет фиксированных эффектов для интерпретации.")
        return

    print("\nИнтерпретация фиксированных эффектов:")

    for _, row in fixed_df.iterrows():
        level = parse_term_level(row["term"])
        p_pos = row["post_prob_gt_0"]
        p_neg = row["post_prob_lt_0"]
        p_rope = row["post_prob_in_rope"]

        if p_pos >= 0.95:
            conclusion = "с высокой апостериорной вероятностью модель лучше baseline"
        elif p_neg >= 0.95:
            conclusion = "с высокой апостериорной вероятностью модель хуже baseline"
        elif p_rope >= 0.5:
            conclusion = "эффект, вероятно, практически мал"
        else:
            conclusion = "данные не дают уверенного вывода"

        print(
            f"  {level}: median OR = {row['median_odds_ratio']:.3f}, "
            f"{int(CREDIBLE_INTERVAL*100)}% HDI OR "
            f"[{row['hdi_low_odds_ratio']:.3f}; {row['hdi_high_odds_ratio']:.3f}], "
            f"P(beta>0) = {p_pos:.3f}, "
            f"P(beta in ROPE) = {p_rope:.3f} -> {conclusion}"
        )

def interpret_pairwise_comparisons(pairwise_df: pd.DataFrame):
    """
    Печатает понятную интерпретацию попарных сравнений.
    """
    if pairwise_df.empty:
        print("Нет попарных сравнений для интерпретации.")
        return

    print("\nИнтерпретация попарных сравнений:")

    for _, row in pairwise_df.iterrows():
        p_ab = row["post_prob_a_gt_b"]
        p_ba = row["post_prob_b_gt_a"]
        p_rope = row["post_prob_in_rope"]

        if p_ab >= 0.95:
            conclusion = "первая модель, вероятно, лучше второй"
        elif p_ba >= 0.95:
            conclusion = "вторая модель, вероятно, лучше первой"
        elif p_rope >= 0.5:
            conclusion = "различие, вероятно, практически мало"
        else:
            conclusion = "убедительного различия не видно"

        print(
            f"  {row['comparison']}: median OR = {row['median_odds_ratio']:.3f}, "
            f"{int(CREDIBLE_INTERVAL*100)}% HDI OR "
            f"[{row['hdi_low_odds_ratio']:.3f}; {row['hdi_high_odds_ratio']:.3f}], "
            f"P(first > second) = {p_ab:.3f}, "
            f"P(in ROPE) = {p_rope:.3f} -> {conclusion}"
        )

def build_descriptive_tables(df: pd.DataFrame, output_dir: str):
    """
    Простые описательные таблицы по моделям.

    Для каждого критерия:
    - находим самое частое значение отдельно для каждой модели;
    - исключаем строки, где значение критерия равно этому модальному значению;
    - по отфильтрованному df считаем median и mean для этого критерия;
    - строим distribution без этого модального значения.
    """
    models = pd.Index(pd.unique(df["model"]))

    def get_model_mode(series: pd.Series):
        modes = series.mode(dropna=False)
        return modes.iloc[0] if not modes.empty else pd.NA

    median_cols = []
    mean_cols = []

    for criterion in CRITERIA:
        model_modes = (
            df.groupby("model")[criterion]
            .agg(get_model_mode)
            .rename(f"{criterion}_mode")
            .reset_index()
        )

        criterion_df = (
            df.merge(model_modes, on="model", how="left")
            .loc[lambda x: x[criterion] != x[f"{criterion}_mode"]]
            .copy()
        )

        median_by_model = (
            criterion_df.groupby("model")[criterion]
            .median()
            .reindex(models)
            .rename(criterion)
        )

        mean_by_model = (
            criterion_df.groupby("model")[criterion]
            .mean()
            .round(3)
            .reindex(models)
            .rename(criterion)
        )

        median_cols.append(median_by_model)
        mean_cols.append(mean_by_model)

        dist = (
            criterion_df.groupby(["model", criterion])
            .size()
            .reset_index(name="count")
            .sort_values(["model", criterion])
        )
        filename = f"distribution_{criterion}.csv".replace("/", "_")
        dist.to_csv(os.path.join(output_dir, filename), index=False)

    median_table = pd.concat(median_cols, axis=1)
    mean_table = pd.concat(mean_cols, axis=1)

    median_table.to_csv(os.path.join(output_dir, "descriptives_median_by_model.csv"))
    mean_table.to_csv(os.path.join(output_dir, "descriptives_mean_by_model.csv"))

def run_analysis_for_one_criterion(df: pd.DataFrame, criterion: str, output_dir: str) -> Dict[str, pd.DataFrame]:
    """
    Полный анализ одного критерия.
    """
    sub = clean_data_for_criterion(df, criterion)
    if len(sub) == 0:
        print(f"[WARN] Нет валидных данных для критерия {criterion}")
        return {}

    # Полезно удалить неиспользуемые категории
    sub["model"] = sub["model"].cat.remove_unused_categories()
    sub["dialog"] = sub["dialog"].cat.remove_unused_categories()
    sub["session_id"] = sub["session_id"].cat.remove_unused_categories()

    # Обучение модели
    model, idata = fit_bayesian_ordinal_model(sub, criterion)

    # Сохраняем NetCDF с posterior
    netcdf_path = os.path.join(output_dir, f"posterior_{criterion}.nc")
    idata.to_netcdf(netcdf_path)

    # Сводка
    trace_summary_path = os.path.join(output_dir, f"trace_summary_{criterion}.csv")
    trace_summary = save_trace_summary(idata, trace_summary_path)

    fixed_df = summarize_fixed_effects(idata, criterion)
    pairwise_df = pairwise_model_comparisons_from_posterior(idata, sub, criterion)
    random_df = random_effects_summary(idata, criterion)
    diag_df = diagnostics_table(idata, criterion)
    ppc_df = posterior_predictive_summary(model, idata, sub, criterion)

    fixed_df.to_csv(os.path.join(output_dir, f"fixed_effects_{criterion}.csv"), index=False)
    pairwise_df.to_csv(os.path.join(output_dir, f"pairwise_comparisons_{criterion}.csv"), index=False)
    random_df.to_csv(os.path.join(output_dir, f"random_effects_{criterion}.csv"), index=False)
    diag_df.to_csv(os.path.join(output_dir, f"diagnostics_{criterion}.csv"), index=False)
    ppc_df.to_csv(os.path.join(output_dir, f"posterior_predictive_{criterion}.csv"), index=False)

    print(f"\nСохранены файлы для критерия {criterion}:")
    print(f"  {netcdf_path}")
    print(f"  {trace_summary_path}")
    print(f"  {os.path.join(output_dir, f'fixed_effects_{criterion}.csv')}")
    print(f"  {os.path.join(output_dir, f'pairwise_comparisons_{criterion}.csv')}")
    print(f"  {os.path.join(output_dir, f'random_effects_{criterion}.csv')}")
    print(f"  {os.path.join(output_dir, f'diagnostics_{criterion}.csv')}")
    print(f"  {os.path.join(output_dir, f'posterior_predictive_{criterion}.csv')}")

    # Печать интерпретации
    interpret_fixed_effects(fixed_df)
    interpret_pairwise_comparisons(pairwise_df)

    return {
        "trace_summary": trace_summary,
        "fixed_effects": fixed_df,
        "pairwise": pairwise_df,
        "random_effects": random_df,
        "diagnostics": diag_df,
        "ppc": ppc_df
    }

def combine_all_results(all_results: Dict[str, Dict[str, pd.DataFrame]], output_dir: str):
    """
    Склеивает результаты по всем критериям в общие CSV.
    """
    keys = ["fixed_effects", "pairwise", "random_effects", "diagnostics", "ppc"]
    for key in keys:
        dfs = []
        for criterion, res in all_results.items():
            if key in res and isinstance(res[key], pd.DataFrame) and not res[key].empty:
                dfs.append(res[key])
        if dfs:
            combined = pd.concat(dfs, ignore_index=True)
            combined.to_csv(os.path.join(output_dir, f"all_{key}.csv"), index=False)

def print_global_summary(all_results: Dict[str, Dict[str, pd.DataFrame]]):
    """
    Краткий итог по всем критериям.
    """
    print_header("ИТОГОВАЯ СВОДКА ПО ВСЕМ КРИТЕРИЯМ")

    for criterion, res in all_results.items():
        print(f"\nКритерий: {criterion}")

        fixed_df = res.get("fixed_effects", pd.DataFrame())
        pairwise_df = res.get("pairwise", pd.DataFrame())
        diag_df = res.get("diagnostics", pd.DataFrame())

        if not diag_df.empty and "r_hat" in diag_df.columns:
            max_rhat = diag_df["r_hat"].dropna().max()
            print(f"  max R-hat: {max_rhat:.4f}")
            if max_rhat > 1.01:
                print("  [WARN] Возможны проблемы со сходимостью.")
            else:
                print("  Сходимость по R-hat выглядит хорошей.")

        if not fixed_df.empty:
            strong = fixed_df[
                (fixed_df["post_prob_gt_0"] >= 0.95) |
                (fixed_df["post_prob_lt_0"] >= 0.95)
            ]
            print(f"  Число выраженных отличий от baseline: {len(strong)}")

        if not pairwise_df.empty:
            strong_pw = pairwise_df[
                (pairwise_df["post_prob_a_gt_b"] >= 0.95) |
                (pairwise_df["post_prob_b_gt_a"] >= 0.95)
            ]
            print(f"  Число убедительных попарных различий: {len(strong_pw)}")

In [24]:
def main_mixed():
    ensure_output_dir(OUTPUT_DIR)

    print_header("ЗАГРУЗКА ДАННЫХ")
    df = load_and_prepare_data(CSV_PATH)

    describe_dataset(df)
    build_descriptive_tables(df, OUTPUT_DIR)

    all_results = {}

    for criterion in CRITERIA:
        res = run_analysis_for_one_criterion(df, criterion, OUTPUT_DIR)
        all_results[criterion] = res

    combine_all_results(all_results, OUTPUT_DIR)
    print_global_summary(all_results)

    print_header("ГОТОВО")
    print("Байесовский ordinal mixed-effects анализ завершен.")
    print("Посмотрите CSV-файлы и .nc-файлы в папке с результатами.")

# Запуск

In [34]:
ensure_output_dir(OUTPUT_DIR)

print_header("ЗАГРУЗКА ДАННЫХ")
df = load_and_prepare_data(CSV_PATH)

describe_dataset(df)

# Описательные таблицы
build_descriptive_tables(df, OUTPUT_DIR)


ЗАГРУЗКА ДАННЫХ

ОПИСАНИЕ ДАННЫХ
Число строк: 179
Число оценщиков (session_id): 17
Число моделей: 4
Число диалогов: 13

Модели:
  - claude-haiku-4-5-automata_v2-delivery_v2
  - deepseek-v3.2-automata_v2-delivery_v2
  - gemini-3-flash-preview-automata_v2-delivery_v2
  - gpt-5.4-automata_v2-delivery_v2

Число наблюдений по моделям:
model
claude-haiku-4-5-automata_v2-delivery_v2          44
deepseek-v3.2-automata_v2-delivery_v2             45
gemini-3-flash-preview-automata_v2-delivery_v2    46
gpt-5.4-automata_v2-delivery_v2                   44
Name: count, dtype: int64

Число наблюдений по диалогам:
dialog
quest1     29
quest10    17
quest11     9
quest12     9
quest13    10
quest14    16
quest15    24
quest3      7
quest4     18
quest5     10
quest6     12
quest7      8
quest9     10
Name: count, dtype: int64


In [12]:
# Основной вариант
main_global, main_coef, main_pairwise = run_main_analysis(
    df,
    OUTPUT_DIR,
    include_dialog=USE_DIALOG_FIXED_EFFECT
)


ОСНОВНОЙ ВАРИАНТ АНАЛИЗА
Модель: Ordered Logistic Regression
Учитывать dialog как фиксированный эффект: True

--- Анализ критерия: relevancy ---
Глобальный LR test: LR = 16.6573, p = 0.0008312

--- Анализ критерия: coherence ---
Глобальный LR test: LR = 10.3287, p = 0.01597

--- Анализ критерия: naturalness ---
Глобальный LR test: LR = 7.8182, p = 0.04992

--- Анализ критерия: faithfulness ---
Глобальный LR test: LR = 12.5197, p = 0.005799

--- Анализ критерия: safety ---
Глобальный LR test: LR = 2.3692, p = 0.4994

--- Анализ критерия: unbiasedness ---
Глобальный LR test: LR = 0.2649, p = 0.9665

--- Анализ критерия: non-toxicity ---
Глобальный LR test: LR = 0.6368, p = 0.888

Результаты сохранены:
  ./results\main_global_tests.csv
  ./results\main_model_coefficients.csv
  ./results\main_pairwise_comparisons.csv

ИНТЕРПРЕТАЦИЯ ОСНОВНОГО ВАРИАНТА

Критерий: relevancy
  Есть статистически значимый общий эффект модели (p = 0.0008312).
  После попарных сравнений с поправкой Holm значимых

In [20]:
main_global

,criterion,n_obs,lr_stat,global_p,success,global_p_holm_across_criteria
0,relevancy,179,16.657284,0.000831,True,0.005818
1,coherence,179,10.328720,0.015969,True,0.079845
2,naturalness,179,7.818214,0.049922,True,0.199688
3,faithfulness,179,12.519666,0.005799,True,0.034796
4,safety,179,2.369220,0.499390,True,1.000000
5,unbiasedness,179,0.264918,0.966485,True,1.000000
6,non-toxicity,179,0.636764,0.887967,True,1.000000


In [21]:
main_coef

,criterion,term,beta,se,odds_ratio,ci_low,ci_high,p_value
0,relevancy,model_deepseek-v3.2-automata_v2-delivery_v2,2.653652,0.963167,14.205827,2.150884,93.824474,0.005867
1,relevancy,model_gemini-3-flash-preview-automata_v2-deliv...,1.339884,0.523898,3.818602,1.367607,10.662213,0.010542
2,relevancy,model_gpt-5.4-automata_v2-delivery_v2,2.517360,0.693143,12.395828,3.186188,48.225818,0.000281
3,coherence,model_deepseek-v3.2-automata_v2-delivery_v2,1.601442,0.856024,4.960180,0.926507,26.554983,0.061374
4,coherence,model_gemini-3-flash-preview-automata_v2-deliv...,1.548791,0.523663,4.705779,1.686120,13.133319,0.003100
5,coherence,model_gpt-5.4-automata_v2-delivery_v2,0.937724,0.570922,2.554162,0.834216,7.820211,0.100492
6,naturalness,model_deepseek-v3.2-automata_v2-delivery_v2,1.490990,0.863958,4.441489,0.816820,24.150768,0.084389
7,naturalness,model_gemini-3-flash-preview-automata_v2-deliv...,1.333078,0.517575,3.792698,1.375269,10.459448,0.010006
8,naturalness,model_gpt-5.4-automata_v2-delivery_v2,0.861674,0.582731,2.367121,0.755438,7.417236,0.139225
9,faithfulness,model_deepseek-v3.2-automata_v2-delivery_v2,1.903538,0.888429,6.709593,1.176155,38.276125,0.032146


In [22]:
main_pairwise

,criterion,comparison,beta,se,odds_ratio,ci_low,ci_high,p_value,success,p_holm
0,relevancy,deepseek-v3.2-automata_v2-delivery_v2 vs claud...,1.760583,NaN,5.815828,NaN,NaN,NaN,True,NaN
1,relevancy,gemini-3-flash-preview-automata_v2-delivery_v2...,1.383791,NaN,3.990001,NaN,NaN,NaN,True,NaN
2,relevancy,gpt-5.4-automata_v2-delivery_v2 vs claude-haik...,3.138483,NaN,23.068854,NaN,NaN,NaN,True,NaN
3,relevancy,gemini-3-flash-preview-automata_v2-delivery_v2...,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
4,relevancy,gpt-5.4-automata_v2-delivery_v2 vs deepseek-v3...,-0.140120,NaN,0.869254,NaN,NaN,NaN,True,NaN
5,relevancy,gpt-5.4-automata_v2-delivery_v2 vs gemini-3-fl...,0.792329,NaN,2.208535,NaN,NaN,NaN,True,NaN
6,coherence,deepseek-v3.2-automata_v2-delivery_v2 vs claud...,0.734304,NaN,2.084031,NaN,NaN,NaN,True,NaN
7,coherence,gemini-3-flash-preview-automata_v2-delivery_v2...,1.552443,NaN,4.722993,NaN,NaN,NaN,True,NaN
8,coherence,gpt-5.4-automata_v2-delivery_v2 vs claude-haik...,1.456649,NaN,4.291556,NaN,NaN,NaN,True,NaN
9,coherence,gemini-3-flash-preview-automata_v2-delivery_v2...,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN


In [29]:
main_mixed()


ЗАГРУЗКА ДАННЫХ

ОПИСАНИЕ ДАННЫХ
Число строк: 179
Число оценщиков (session_id): 17
Число моделей: 4
Число диалогов: 13

Модели:
  - claude-haiku-4-5-automata_v2-delivery_v2
  - deepseek-v3.2-automata_v2-delivery_v2
  - gemini-3-flash-preview-automata_v2-delivery_v2
  - gpt-5.4-automata_v2-delivery_v2

Число наблюдений по моделям:
model
claude-haiku-4-5-automata_v2-delivery_v2          44
deepseek-v3.2-automata_v2-delivery_v2             45
gemini-3-flash-preview-automata_v2-delivery_v2    46
gpt-5.4-automata_v2-delivery_v2                   44
Name: count, dtype: int64

Число наблюдений по диалогам:
dialog
quest1     29
quest10    17
quest11     9
quest12     9
quest13    10
quest14    16
quest15    24
quest3      7
quest4     18
quest5     10
quest6     12
quest7      8
quest9     10
Name: count, dtype: int64

ОБУЧЕНИЕ МОДЕЛИ ДЛЯ КРИТЕРИЯ: relevancy
Уровни model:
  - claude-haiku-4-5-automata_v2-delivery_v2  [baseline]
  - deepseek-v3.2-automata_v2-delivery_v2
  - gemini-3-flash-prev

IndexError: tuple index out of range

In [35]:
simplified_global, simplified_pairwise = run_simplified_analysis(
    df,
    OUTPUT_DIR,
    agg_func="median"   # можно заменить на "mean"
)


УПРОЩЕННЫЙ ВАРИАНТ АНАЛИЗА
Агрегация session_id x model: median

--- Анализ критерия: relevancy ---
Friedman test: chi2 = 3.9000, p = 0.2725, Kendall's W = 0.325, n = 4

--- Анализ критерия: coherence ---
Friedman test: chi2 = 4.6000, p = 0.2035, Kendall's W = 0.192, n = 8

--- Анализ критерия: naturalness ---
Friedman test: chi2 = 0.1139, p = 0.9901, Kendall's W = 0.004, n = 10

--- Анализ критерия: faithfulness ---
Friedman test: chi2 = 7.8750, p = 0.04867, Kendall's W = 0.263, n = 10

--- Анализ критерия: safety ---
Friedman test: chi2 = 3.0000, p = 0.3916, Kendall's W = 0.500, n = 2

--- Анализ критерия: unbiasedness ---
Не удалось выполнить Friedman test: Недостаточно оценщиков после агрегации.

--- Анализ критерия: non-toxicity ---
Friedman test: chi2 = 1.0541, p = 0.7882, Kendall's W = 0.070, n = 5

Результаты сохранены:
  ./results_no_ten\simplified_global_tests.csv
  ./results_no_ten\simplified_pairwise_comparisons.csv

ИНТЕРПРЕТАЦИЯ УПРОЩЕННОГО ВАРИАНТА

Критерий: relevancy


In [36]:
simplified_global

,criterion,stat,p_value,kendalls_w,n_subjects,k_models,success,p_holm_across_criteria
0,relevancy,3.900000,0.272467,0.325000,4.0,4.0,True,1.000000
1,coherence,4.600000,0.203542,0.191667,8.0,4.0,True,1.000000
2,naturalness,0.113924,0.990116,0.003797,10.0,4.0,True,1.000000
3,faithfulness,7.875000,0.048667,0.262500,10.0,4.0,True,0.292002
4,safety,3.000000,0.391625,0.500000,2.0,4.0,True,1.000000
5,unbiasedness,NaN,NaN,NaN,NaN,NaN,False,NaN
6,non-toxicity,1.054054,0.788176,0.070270,5.0,4.0,True,1.000000


In [37]:
parts = simplified_pairwise["comparison"].str.split(r"\s+vs\s+", n=1, expand=True)

simplified_pairwise["comparison_a"] = parts[0].str.strip()
simplified_pairwise["comparison_b"] = parts[1].str.strip()

simplified_pairwise = simplified_pairwise.drop(columns=["comparison"])

In [38]:
simplified_pairwise

,criterion,n_subjects,wilcoxon_stat,p_value,rank_biserial,median_model_1,median_model_2,mean_model_1,mean_model_2,p_holm,comparison_a,comparison_b
0,relevancy,4,1.5,0.750000,0.500000,6.00,5.25,5.5000,5.2500,1.000000,claude-haiku-4-5-automata_v2-delivery_v2,deepseek-v3.2-automata_v2-delivery_v2
1,relevancy,4,0.0,1.000000,1.000000,6.00,5.50,5.5000,5.2500,1.000000,claude-haiku-4-5-automata_v2-delivery_v2,gemini-3-flash-preview-automata_v2-delivery_v2
2,relevancy,4,0.0,0.500000,-1.000000,6.00,6.25,5.5000,5.8750,1.000000,claude-haiku-4-5-automata_v2-delivery_v2,gpt-5.4-automata_v2-delivery_v2
3,relevancy,4,3.0,1.000000,0.000000,5.25,5.50,5.2500,5.2500,1.000000,deepseek-v3.2-automata_v2-delivery_v2,gemini-3-flash-preview-automata_v2-delivery_v2
4,relevancy,4,1.5,0.375000,-0.700000,5.25,6.25,5.2500,5.8750,1.000000,deepseek-v3.2-automata_v2-delivery_v2,gpt-5.4-automata_v2-delivery_v2
5,relevancy,4,0.0,0.250000,-1.000000,5.50,6.25,5.2500,5.8750,1.000000,gemini-3-flash-preview-automata_v2-delivery_v2,gpt-5.4-automata_v2-delivery_v2
6,coherence,8,10.5,0.328125,0.416667,9.25,8.50,8.1250,7.2500,1.000000,claude-haiku-4-5-automata_v2-delivery_v2,deepseek-v3.2-automata_v2-delivery_v2
7,coherence,8,1.5,0.093750,0.857143,9.25,8.25,8.1250,7.3125,0.562500,claude-haiku-4-5-automata_v2-delivery_v2,gemini-3-flash-preview-automata_v2-delivery_v2
8,coherence,8,11.0,0.359375,0.388889,9.25,8.00,8.1250,7.1250,1.000000,claude-haiku-4-5-automata_v2-delivery_v2,gpt-5.4-automata_v2-delivery_v2
9,coherence,8,9.5,0.875000,0.095238,8.50,8.25,7.2500,7.3125,1.000000,deepseek-v3.2-automata_v2-delivery_v2,gemini-3-flash-preview-automata_v2-delivery_v2


In [ ]:
print_header("ГОТОВО")
print("Анализ завершен.")
print("Посмотрите CSV-файлы в папке с результатами.")
print("\nРекомендуемая интерпретация:")
print("1. Сначала смотрите main_global_tests.csv")
print("2. Затем main_pairwise_comparisons.csv")
print("3. Для проверки устойчивости — simplified_global_tests.csv и simplified_pairwise_comparisons.csv")